# Hand-in-eye 数据采集：连接机器人和相机

这个 notebook 的任务不是求解标定结果，而是采集后续标定需要的数据集。

每一帧数据应该包含：RGB 图、depth 图、相机内参、机器人末端位姿、以及可选的关节角。后续 `hand_in_eye_calib.py` 会把这些数据喂给 OpenCV 的 hand-eye calibration。

当前版本强依赖 Franka/Panda：`panda_py.Panda` 负责连接和运动控制，`panda.move_to_pose()` 负责笛卡尔空间移动，`panda.get_log()["q"]` 负责读取关节角。迁移到 ViperX 时，优先把这些操作封装成一个小 adapter：`move_to_pose()`、`get_joint_positions()`、`get_hand_pose()`，这样后面的采集逻辑可以尽量保持不变。

In [10]:
# -----------------------------
# 1. Robot connection settings
# -----------------------------
# Original robot: Franka/Panda. ``hostname`` is the robot controller IP used by
# panda-py/libfranka.
#
# ViperX replacement point:
# - Replace these connection fields with whatever your ViperX/LeRobot stack
#   needs, such as serial port, USB device path, CAN interface, or robot ID.
# - Keep ``base_dir`` because the calibration scripts expect the same folder
#   layout regardless of robot brand.
hostname = '172.16.0.2'
username = 'TQSH'
password = 'tqshanghai'
base_dir = '../../data/hand_in_eye8'

# panda-py prints useful controller state and errors through Python logging.
# For ViperX, configure your driver logs here if needed.
import logging
import panda_py

logging.basicConfig(level=logging.INFO)

In [11]:
# libfranka provides the low-level gripper interface for Franka/Panda.
# The gripper object is created here but not used later in this notebook.
# You can remove or replace it when porting to ViperX if your camera is wrist
# mounted and gripper state is irrelevant to calibration.
from panda_py import libfranka

# Create the Panda robot handle. This object owns motion commands such as
# ``move_to_pose`` and logging calls such as ``get_log``.
#
# ViperX replacement point:
# - This should become your ViperX hardware/controller object, for example a
#   LeRobot robot instance or your own wrapper around it.
panda = panda_py.Panda(hostname)

# Enable state logging at about 100 Hz. Later we read the most recent joint
# vector with ``panda.get_log()["q"][-1]``.
#
# ViperX replacement point:
# - Make sure your controller can provide joint positions at the exact moment
#   each image is captured. If timestamps are available, save them too.
panda.enable_logging(int(1e2))
gripper = libfranka.Gripper(hostname)

INFO:panda:Connected to robot (172.16.0.2).
INFO:panda:Panda class destructor invoked (172.16.0.2).
INFO:panda:Stopping active controller (Trajectory).


# 采集函数和坐标系约定

Hand-in-eye 的意思是相机固定在机器人手腕或末端附近，因此相机坐标系和末端坐标系之间存在一个固定变换 `T_camera_to_hand`。采集时要让机器人移动到多个不同姿态，让相机从不同角度看到同一块 ChArUco 标定板。

这一段代码做的是采样动作，不做最终求解。最重要的数据约定是：每张图片都要对应同一时刻的机器人手部位姿 `T_hand_to_base` 或足够恢复它的关节角。Franka 版本通过 `panda.move_to_pose()` 和 `panda.get_log()["q"]` 实现；ViperX 版本应当用你的控制接口和 FK 模型提供同样语义的数据。

In [12]:
# Franka/Panda predefined joint positions and FK helper. The later code uses
# ``constants.JOINT_POSITION_START`` as a safe nominal pose and calls
# ``panda_py.fk`` to get the corresponding 4x4 Cartesian pose.
#
# ViperX replacement point:
# - Replace this with your own start joint vector and ViperX FK result.
# - If you use roboticstoolbox, this is where a ViperX model's FK output should
#   enter the same 4x4 homogeneous-transform convention.
from panda_py import constants

# SciPy Rotation is used only to build small roll/pitch/yaw perturbations.
# The motion commands still consume a 4x4 pose matrix.
from scipy.spatial.transform import Rotation as R
import numpy as np
import matplotlib.pyplot as plt
import os
import time

In [13]:
def generate_and_move_to_pose(init_pose, roll, pitch, yaw, z_add, x_add, y_add, max_roll_deviation, max_pitch_deviation, max_yaw_deviation):
    """Generate one calibration pose, command the robot, and return that pose.

    Concept:
    - A good hand-eye dataset needs multiple robot poses with sufficiently
      different rotations and translations. If all frames are too similar, the
      calibration problem becomes poorly constrained.
    - This function starts from ``init_pose`` and applies a nominal Euler-angle
      offset plus a small random perturbation. That random perturbation is the
      original code's simple way to get pose diversity.

    Coordinate convention:
    - ``init_pose`` is a 4x4 homogeneous transform. For the original Panda path
      it is generated by ``panda_py.fk(constants.JOINT_POSITION_START)``.
    - The upper-left 3x3 block is rotation. The last column stores x/y/z
      translation in meters.
    - ``roll``, ``pitch``, and ``yaw`` are radians because ``degrees=False``.

    ViperX replacement point:
    - Keep this function's output contract if possible: return the actual or
      commanded 4x4 hand/TCP pose for the captured frame.
    - Replace ``panda.move_to_pose(pose)`` with your LeRobot/ViperX motion
      command. If your controller works in joint space, solve IK first and then
      command joints; if it works in Cartesian space, send the pose directly.
    - After motion, consider reading back the measured pose/joints instead of
      trusting the commanded pose, especially for lower-cost arms with backlash
      or imperfect tracking.
    """
    # Add random orientation perturbations around the nominal roll/pitch/yaw.
    # These random offsets should remain small enough to keep the board in view
    # and avoid self-collision or joint limits.
    roll_turbulent = roll + np.random.uniform(-max_roll_deviation, max_roll_deviation)
    pitch_turbulent = pitch + np.random.uniform(-max_pitch_deviation, max_pitch_deviation)
    yaw_turbulent = yaw + np.random.uniform(-max_yaw_deviation, max_yaw_deviation)

    # Convert Euler angles into a 3x3 rotation matrix. The sequence 'xyz' means
    # roll about X, pitch about Y, and yaw about Z in SciPy's convention.
    r = R.from_euler('xyz', [roll_turbulent, pitch_turbulent, yaw_turbulent], degrees=False)
    rotation_matrix = r.as_matrix()

    # Compose the new local perturbation with the initial pose orientation.
    # Matrix multiplication order matters. Here the perturbation is applied
    # relative to the initial end-effector orientation.
    absolute_rotation_matrix = np.dot(init_pose[:3, :3], rotation_matrix)

    # Copy the initial pose and then overwrite rotation plus translation. The
    # z/x/y offsets are simple Cartesian translations in meters in the base
    # frame used by the original Panda command.
    pose = init_pose.copy()
    pose[:3, :3] = absolute_rotation_matrix
    pose[2, 3] += z_add
    pose[0, 3] += x_add
    pose[1, 3] += y_add

    # Franka-specific motion command. This is the main hardware-control line to
    # replace for ViperX.
    panda.move_to_pose(pose)
    
    return pose

def save_pose(pose, base_dir, frame_num):
    """Save one 4x4 pose matrix for the matching image frame.

    The filename index must match the RGB/depth frame index. The calibration
    code later sorts filenames lexicographically and pairs list elements by
    order, so keeping a clean one-to-one mapping is important.

    ViperX note:
    - Save the pose only if you know exactly which frame it represents. If your
      adapter records only joints, saving joints is enough because FK can
      reconstruct ``T_hand_to_base`` later.
    """
    pose_filename = f'{base_dir}/poses/pose_{frame_num}.npy'
    np.save(pose_filename, pose)
    print(f"Saved pose to {pose_filename}")

def save_joints(joints, base_dir, frame_num):
    """Save one joint vector for the matching image frame.

    Why joints are useful:
    - FK from measured joints is often more reliable than a commanded Cartesian
      pose, because it reflects what the robot actually reported.
    - ``hand_in_eye_calib.py`` prefers ``joints/`` when present and converts
      joints to a hand pose with ``joint_to_hand``.

    ViperX note:
    - Ensure joint order and units match your ViperX FK model. A common source
      of wrong calibration is mixing degrees/radians or using a different joint
      order from the URDF/roboticstoolbox model.
    """
    joints_filename = f'{base_dir}/joints/joints_{frame_num}.npy'
    np.save(joints_filename, joints)
    print(f"Saved joints to {joints_filename}")


In [14]:
import sys

# Make the local ``realsense`` package importable from this notebook location.
# This mirrors the script style used elsewhere in the repository.
parent_dir = os.path.dirname(os.getcwd())
parent_dir = os.path.dirname(parent_dir)
sys.path.append(parent_dir)

# RealSense helper wrapper. ``Camera`` owns stream start/stop and image capture;
# ``get_devices`` enumerates connected camera serial numbers.
from realsense.realsense import Camera
from realsense.realsense import get_devices


def capture_images(camera, delay_before_shooting, start_frame, picture_nums, base_dir, init_pose, roll, pitch, yaw, z_add, x_add, y_add,
                   max_roll_deviation, max_pitch_deviation, max_yaw_deviation):
    """Move through several calibration poses and save synchronized samples.

    Output dataset written under ``base_dir``:
    - ``rgb/<frame>.png``: RGB image where the ChArUco board should be visible.
    - ``depth/<frame>.npy``: depth image, mainly useful for visual checks and
      TSDF/point-cloud debugging.
    - ``poses/pose_<frame>.npy``: commanded 4x4 pose from this notebook.
    - ``joints/joints_<frame>.npy``: latest logged Panda joint vector.
    - ``rgb_intrinsics.npz`` and ``depth_intrinsics.npz``: RealSense intrinsics.

    Synchronization concept:
    - The calibration assumes each RGB image and robot pose describe the same
      instant. This original code moves first, shoots, then reads the latest
      robot log. For ViperX, it is worth adding timestamps if your stack allows
      it, then pairing image and joint samples by time.

    ViperX replacement point:
    - Replace only the robot-control/readback lines. The camera save format can
      stay unchanged so ``hand_in_eye_calib.py`` continues to work.
    """
    
    # Start RealSense streams before requesting intrinsics or frames.
    camera.start()
    
    # Intrinsics describe how 3D camera rays project into pixel coordinates.
    # Distortion coefficients describe lens distortion. The hand-eye solve uses
    # RGB intrinsics because the ChArUco board is detected in RGB images.
    rgb_intrinsics, rgb_coeffs, depth_intrinsics, depth_coeffs = camera.get_intrinsics_raw()
    depth_scale = camera.get_depth_scale()

    print(f"RGB Intrinsics: {rgb_intrinsics}")
    print(f"RGB Distortion Coefficients: {rgb_coeffs}")
    rgb_intrinsics_path = f'{base_dir}/rgb_intrinsics.npz'
    # Save only the fields later reconstructed by ``calibration.utils.read_data``.
    np.savez(rgb_intrinsics_path, fx=rgb_intrinsics.fx, fy=rgb_intrinsics.fy, ppx=rgb_intrinsics.ppx, ppy=rgb_intrinsics.ppy, coeffs=rgb_intrinsics.coeffs)

    print(f"Depth Scale: {depth_scale}")
    print(f"Depth Intrinsics: {depth_intrinsics}")
    print(f"Depth Distortion Coefficients: {depth_coeffs}")
    depth_intrinsics_path = f'{base_dir}/depth_intrinsics.npz'
    # ``depth_scale`` converts raw depth units to meters. For many RealSense
    # devices this is about 0.001, meaning raw value 1000 is roughly 1 meter.
    np.savez(depth_intrinsics_path, fx=depth_intrinsics.fx, fy=depth_intrinsics.fy, ppx=depth_intrinsics.ppx, ppy=depth_intrinsics.ppy, coeffs=depth_intrinsics.coeffs, depth_scale=depth_scale)

    # Drop one frame and wait so auto-exposure/stream buffers settle before
    # saving calibration images.
    _, _ = camera.shoot()  
    time.sleep(delay_before_shooting)

    # Capture ``picture_nums`` frames for this region of the workspace.
    # ``start_frame`` lets multiple configurations write unique frame numbers.
    for frame_num in range(start_frame, start_frame + picture_nums):
        # Generate a pose, command the robot, and keep the commanded pose for
        # optional fallback use. For higher accuracy, prefer measured joints.
        pose = generate_and_move_to_pose(init_pose, roll, pitch, yaw, z_add, x_add, y_add,
                                         max_roll_deviation, max_pitch_deviation, max_yaw_deviation)

        # Shoot RGB and depth after the robot move command returns. If your
        # ViperX controller returns before motion fully settles, add an explicit
        # wait-for-still condition here.
        rgb_image, depth_image = camera.shoot()
        rgb_filename = f'{base_dir}/rgb/{frame_num}.png'
        depth_filename = f'{base_dir}/depth/{frame_num}.npy'

        # RGB is saved as PNG for visual inspection and OpenCV board detection;
        # depth is saved as NumPy because it preserves raw numeric values.
        plt.imsave(rgb_filename, rgb_image)
        np.save(depth_filename, depth_image)
        print(f"Saved {rgb_filename}")
        print(f"Saved {depth_filename}")

        save_pose(pose, base_dir, frame_num)

        # Franka-specific joint readback. ``q`` is the Panda joint vector in the
        # panda-py log. This is the second main hardware line to replace.
        #
        # ViperX replacement point:
        # - Use your LeRobot/ViperX state API to read measured joints.
        # - Save radians unless your FK model explicitly expects degrees.
        last_qpos = panda.get_log()["q"][-1]
        save_joints(last_qpos, base_dir, frame_num)

    # Return the Panda to its configured start pose. For ViperX, replace with
    # your own safe home/retract command, or remove if manual recovery is safer.
    panda.move_to_start()
        
    # Always stop camera streams after capture to release the device handle.
    camera.stop()


In [15]:
# -----------------------------
# 4. Create output folders
# -----------------------------
# The later calibration reader expects exactly these subfolder names.
os.makedirs(f'{base_dir}/rgb', exist_ok=True)
os.makedirs(f'{base_dir}/depth', exist_ok=True)
os.makedirs(f'{base_dir}/poses', exist_ok=True)
os.makedirs(f'{base_dir}/joints', exist_ok=True)

# -----------------------------
# 5. Define pose-sampling regions
# -----------------------------
# Each dictionary describes one cluster of calibration poses around the Panda
# start pose. The loop below captures 10 images per dictionary.
#
# Meaning of each field:
# - ``init_pose``: base pose before offsets. Original code gets it from Panda FK.
# - ``roll/pitch/yaw``: nominal orientation offset in radians.
# - ``z_add/x_add/y_add``: nominal translation offset in meters.
# - ``max_*_deviation``: random orientation perturbation range in radians.
#
# ViperX replacement point:
# - Replace ``panda_py.fk(constants.JOINT_POSITION_START)`` with a ViperX FK
#   result for a safe start joint configuration.
# - Re-tune all offsets for your real workspace, arm reach, camera FOV, and
#   printed board position. Do not reuse these Panda offsets blindly.
# - The ``base_dir`` key inside each config is currently not used by the loop;
#   the global ``base_dir`` variable is passed to ``capture_images`` instead.
image_configs = [
    {
        'base_dir': '../hand_in_eye2',
        'init_pose': panda_py.fk(constants.JOINT_POSITION_START),
        'roll': 0.0,
        'pitch': 0.2,
        'yaw': 0.0,
        'z_add': 0.16,
        'x_add': 0.12,
        'y_add': 0.0,
        'max_roll_deviation': 0.1,
        'max_pitch_deviation': 0.1,
        'max_yaw_deviation': 0.1
    },
    {
        'base_dir': '../hand_in_eye2',
        'init_pose': panda_py.fk(constants.JOINT_POSITION_START),
        'roll': 0.3,
        'pitch': 0.35,
        'yaw': 0.0,
        'z_add': 0.1,
        'x_add': 0.0,
        'y_add': -0.25,
        'max_roll_deviation': 0.1,
        'max_pitch_deviation': 0.1,
        'max_yaw_deviation': 0.1
    },
    {
        'base_dir': '../hand_in_eye2',
        'init_pose': panda_py.fk(constants.JOINT_POSITION_START),
        'roll': -0.25,
        'pitch': 0.35,
        'yaw': 0.2,
        'z_add': 0.12,
        'x_add': 0.1,
        'y_add': 0.20,
        'max_roll_deviation': 0.1,
        'max_pitch_deviation': 0.1,
        'max_yaw_deviation': 0.1
    },
    {
        'base_dir': '../hand_in_eye2',
        'init_pose': panda_py.fk(constants.JOINT_POSITION_START),
        'roll': -0.25,
        'pitch': 0.2,
        'yaw': 0.2,
        'z_add': 0.20,
        'x_add': 0.11,
        'y_add': 0.1,
        'max_roll_deviation': 0.1,
        'max_pitch_deviation': 0.1,
        'max_yaw_deviation': 0.1
    }
]


# -----------------------------
# 6. Select and configure RealSense
# -----------------------------
# Enumerate connected RealSense cameras. If several cameras are connected,
# ``device_serials[0]`` may not be the wrist camera you want; choose the serial
# explicitly for reproducible experiments.
device_serials = get_devices()

# Print selected device serial number for the experiment log.
print("Selected device serial numbers:", device_serials[0])

# Resolution is ``(width, height)``. Changing this changes intrinsics, so the
# intrinsics must always be saved from the same stream configuration used for
# capture.
rgb_resolution = (640, 480)
depth_resolution = (640, 480)

camera = Camera(device_serials[0], rgb_resolution, depth_resolution)

# Delay before shooting, in seconds. Increase this if auto exposure needs more
# time or if the robot has visible settling after each motion.
delay_before_shooting = 3.0

# -----------------------------
# 7. Run the capture loop
# -----------------------------
# This loop produces 4 * 10 = 40 frames with the current settings.
# For robust hand-eye calibration, inspect the saved RGB images afterwards:
# the ChArUco board should be sharp, fully or mostly visible, and observed from
# noticeably different wrist orientations.
for i, config in enumerate(image_configs):
    capture_images(camera, delay_before_shooting, 10*i, 10, base_dir, config['init_pose'], config['roll'], config['pitch'], config['yaw'],
                   config['z_add'], config['x_add'], config['y_add'], config['max_roll_deviation'],
                   config['max_pitch_deviation'], config['max_yaw_deviation'])

Selected device serial numbers: 115222071207
RGB Intrinsics: [ 640x480  p[330.552 241.445]  f[604.902 605.103]  Inverse Brown Conrady [0 0 0 0 0] ]
RGB Distortion Coefficients: [0.0, 0.0, 0.0, 0.0, 0.0]
Depth Scale: 0.0010000000474974513
Depth Intrinsics: [ 640x480  p[322.025 240.057]  f[379.037 379.037]  Brown Conrady [0 0 0 0 0] ]
Depth Distortion Coefficients: [0.0, 0.0, 0.0, 0.0, 0.0]


INFO:panda:Initializing motion generation (moveToPose).
INFO:motion:Computed Cartesian trajectory: 1 waypoint, duration 0.60 seconds.
INFO:panda:Starting new controller (Trajectory).
ERROR:panda:Control loop interruped: libfranka: Move command rejected: command not possible in the current mode ("User stopped")!
INFO:panda:Stopping active controller (Trajectory).
INFO:panda:Initializing motion generation (moveToPose).
INFO:motion:Computed Cartesian trajectory: 1 waypoint, duration 0.50 seconds.
INFO:panda:Starting new controller (Trajectory).
ERROR:panda:Control loop interruped: libfranka: Move command rejected: command not possible in the current mode ("User stopped")!
INFO:panda:Stopping active controller (Trajectory).
INFO:panda:Initializing motion generation (moveToPose).
INFO:motion:Computed Cartesian trajectory: 1 waypoint, duration 0.66 seconds.
INFO:panda:Starting new controller (Trajectory).
ERROR:panda:Control loop interruped: libfranka: Move command rejected: command not poss

Saved ../../hand_in_eye8/rgb/0.png
Saved ../../hand_in_eye8/depth/0.npy
Saved pose to ../../hand_in_eye8/poses/pose_0.npy
Saved joints to ../../hand_in_eye8/joints/joints_0.npy
Saved ../../hand_in_eye8/rgb/1.png
Saved ../../hand_in_eye8/depth/1.npy
Saved pose to ../../hand_in_eye8/poses/pose_1.npy
Saved joints to ../../hand_in_eye8/joints/joints_1.npy
Saved ../../hand_in_eye8/rgb/2.png
Saved ../../hand_in_eye8/depth/2.npy
Saved pose to ../../hand_in_eye8/poses/pose_2.npy
Saved joints to ../../hand_in_eye8/joints/joints_2.npy


INFO:panda:Stopping active controller (Trajectory).
INFO:panda:Initializing motion generation (moveToPose).
INFO:motion:Computed Cartesian trajectory: 1 waypoint, duration 0.59 seconds.
INFO:panda:Starting new controller (Trajectory).
ERROR:panda:Control loop interruped: libfranka: Move command rejected: command not possible in the current mode ("User stopped")!
INFO:panda:Stopping active controller (Trajectory).
INFO:panda:Initializing motion generation (moveToPose).
INFO:motion:Computed Cartesian trajectory: 1 waypoint, duration 0.51 seconds.
INFO:panda:Starting new controller (Trajectory).
ERROR:panda:Control loop interruped: libfranka: Move command rejected: command not possible in the current mode ("User stopped")!
INFO:panda:Stopping active controller (Trajectory).
INFO:panda:Initializing motion generation (moveToPose).
INFO:motion:Computed Cartesian trajectory: 1 waypoint, duration 0.50 seconds.
INFO:panda:Starting new controller (Trajectory).
ERROR:panda:Control loop interruped

Saved ../../hand_in_eye8/rgb/3.png
Saved ../../hand_in_eye8/depth/3.npy
Saved pose to ../../hand_in_eye8/poses/pose_3.npy
Saved joints to ../../hand_in_eye8/joints/joints_3.npy
Saved ../../hand_in_eye8/rgb/4.png
Saved ../../hand_in_eye8/depth/4.npy
Saved pose to ../../hand_in_eye8/poses/pose_4.npy
Saved joints to ../../hand_in_eye8/joints/joints_4.npy
Saved ../../hand_in_eye8/rgb/5.png
Saved ../../hand_in_eye8/depth/5.npy
Saved pose to ../../hand_in_eye8/poses/pose_5.npy
Saved joints to ../../hand_in_eye8/joints/joints_5.npy


...

Saved ../../hand_in_eye8/rgb/37.png
Saved ../../hand_in_eye8/depth/37.npy
Saved pose to ../../hand_in_eye8/poses/pose_37.npy
Saved joints to ../../hand_in_eye8/joints/joints_37.npy


INFO:panda:Stopping active controller (Trajectory).
INFO:panda:Initializing motion generation (moveToPose).
INFO:motion:Computed Cartesian trajectory: 1 waypoint, duration 0.47 seconds.
INFO:panda:Starting new controller (Trajectory).


Saved ../../hand_in_eye8/rgb/38.png
Saved ../../hand_in_eye8/depth/38.npy
Saved pose to ../../hand_in_eye8/poses/pose_38.npy
Saved joints to ../../hand_in_eye8/joints/joints_38.npy


INFO:panda:Stopping active controller (Trajectory).
INFO:panda:Initializing motion generation (moveToJointPosition).
INFO:motion:Computed joint trajectory: 1 waypoint, duration 2.67 seconds.
INFO:panda:Starting new controller (Trajectory).


Saved ../../hand_in_eye8/rgb/39.png
Saved ../../hand_in_eye8/depth/39.npy
Saved pose to ../../hand_in_eye8/poses/pose_39.npy
Saved joints to ../../hand_in_eye8/joints/joints_39.npy
